INIT


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

In [0]:
def log(step, message):
    print(f"[{step}] {message}")

Read From Silver Table/s


In [0]:
log("Gold", "Loading Silver data")

df = spark.table("silver.orders")

log("Gold", f"Rows loaded: {df.count()}")
log("Gold", f"Columns available: {df.columns}")

Gold Customer

Business Transformation and Modeling

In [0]:
latest_date = df.agg(F.max("order_date").alias("max_date")).first()["max_date"]

In [0]:
df_flags = (
    df
    .withColumn("is_last_1m",  F.col("order_date") >= F.date_sub(F.lit(latest_date), 30))
    .withColumn("is_last_6m",  F.col("order_date") >= F.date_sub(F.lit(latest_date), 180))
    .withColumn("is_last_12m", F.col("order_date") >= F.date_sub(F.lit(latest_date), 365))
)

In [0]:
customer_df = (
    df_flags
    .groupBy(
        "customer_id",
        "customer_first_name",
        "customer_last_name",
        "segment",
        "country"
    )
    .agg(
        F.sum(F.when(F.col("is_last_1m"), 1).otherwise(0)).alias("orders_last_1_month"),
        F.sum(F.when(F.col("is_last_6m"), 1).otherwise(0)).alias("orders_last_6_months"),
        F.sum(F.when(F.col("is_last_12m"), 1).otherwise(0)).alias("orders_last_12_months"),
        F.count("*").alias("orders_all_time")
    )
)

Write it to Gold Table

In [0]:
customer_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.customer")

In [0]:
customer_df.display()

Gold Sales


In [0]:
sales_df = df.select(
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "city"
)


In [0]:
sales_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold.sales")

In [0]:
sales_df.display()